In [2]:
import pandas as pd
import json
from scipy.stats import mannwhitneyu
import numpy as np
import sys;sys.path.append("../networks")
from utils import resolve_paths

READ_DATA_PATHS, WRITE_DATA_PATHS = resolve_paths(read_datasets=["Results Data"], 
                                                write_datasets=["Results Data"])

Loading DATA_PATHS.yaml from ../../data/DATA_PATHS.yaml


In [3]:

ablations = ['Ablation - COI', 'Ablation - ECI', 'Ablation - # Prod', "Ablation - SRCA", "Ablation - Geo-Positional", 'Ablation - HHI', "Ablation - TI", 
             'Ablation - Export Value', 'Ablation - Avg.PCI', 'Ablation - Trade Agreements', 'Ablation - Trustworthiness']
models = ['GCN', "GAT", "SAGE"]
graphs = ["export", "total", "export-layered", "multi-graph-total", "multi-graph-export"]

baseline = 'No Ablation'

results = {
    "Graph": [],
    "Method": [],
    "Ablated measure": [],
    "Seed": [],
    "F1 - Positives": []
}

# Read the baseline (No Ablation) results
for model in models:

    for graph in graphs:

        #baseline_results = []

        for seed in range(1, 11):
            
            with open(READ_DATA_PATHS["Results Data"] + f"/{baseline}/{model}/{graph}/{seed}/report.json") as f:
                baseline_result = json.load(f)["F1 - Positives"]
                #baseline_results.append(baseline_result)
                
                results["Graph"].append(graph)
                results["Method"].append(model)
                results["Ablated measure"].append("No Ablation")
                results["Seed"].append(seed)
                results["F1 - Positives"].append(baseline_result)

i = 0

for ablation in ablations:

    for model in models:

        for graph in graphs:

            # These ablations are not applicable to certain graph types
            if ((ablation in ["Ablation - Trustworthiness"]) and (graph in ["export-layered", "total", "export"])) or \
            ((ablation in ["Ablation - SRCA", "Ablation - TI"]) and (graph in ["multi-graph-total", "multi-graph-export"])):
                continue

            ablation_results = []
            baseline_results = []

            for seed in range(1, 11):
                with open(READ_DATA_PATHS["Results Data"] + f"/{ablation}/{model}/{graph}/{seed}/report.json") as f:
                    ablated = json.load(f)["F1 - Positives"]
                    ablation_results.append(ablated)
                
                with open(READ_DATA_PATHS["Results Data"] + f"/{baseline}/{model}/{graph}/{seed}/report.json") as f:
                    base = json.load(f)["F1 - Positives"]
                    baseline_results.append(base)

                results["Graph"].append(graph)
                results["Method"].append(model)
                results["Ablated measure"].append(ablation)
                results["Seed"].append(seed)
                results["F1 - Positives"].append(ablated)
                
                
            stat, pvalue = mannwhitneyu(baseline_results, ablation_results)
            print(
                f"{ablation:<20} | {model:<6} | {graph:<20} | "
                f"Ablation: {np.mean(ablation_results):.4f} | "
                f"Baseline: {np.mean(baseline_results):.4f} | "
                f"Difference: {np.mean(ablation_results) - np.mean(baseline_results):.4f} | "
                f"p-value: {pvalue:.4f} {'(Significant)' if pvalue < (0.05/(len(ablations)*len(models)*len(graphs)-21)) else ''}"
            )
            i += 1

    print()

Ablation - COI       | GCN    | export               | Ablation: 0.1075 | Baseline: 0.1058 | Difference: 0.0018 | p-value: 0.1620 
Ablation - COI       | GCN    | total                | Ablation: 0.0977 | Baseline: 0.1168 | Difference: -0.0190 | p-value: 0.0028 
Ablation - COI       | GCN    | export-layered       | Ablation: 0.1002 | Baseline: 0.1025 | Difference: -0.0023 | p-value: 0.3447 
Ablation - COI       | GCN    | multi-graph-total    | Ablation: 0.9146 | Baseline: 0.9220 | Difference: -0.0074 | p-value: 0.0071 
Ablation - COI       | GCN    | multi-graph-export   | Ablation: 0.9214 | Baseline: 0.9165 | Difference: 0.0049 | p-value: 0.1298 
Ablation - COI       | GAT    | export               | Ablation: 0.1294 | Baseline: 0.1239 | Difference: 0.0055 | p-value: 0.5205 
Ablation - COI       | GAT    | total                | Ablation: 0.1331 | Baseline: 0.1307 | Difference: 0.0024 | p-value: 0.7913 
Ablation - COI       | GAT    | export-layered       | Ablation: 0.1064 | Baseli

In [4]:
results = pd.DataFrame(results)

In [5]:
models = ['GCN', "GAT", "SAGE", "MLP"]
graphs = ["export", "total", "export-layered", "multi-graph-total", "multi-graph-export"]

results = {
    "Graph": [],
    "Method": [],
    "Seed": [],
    "F1 - Positives": []
}

# Read the baseline (No Ablation) results
for model in models:

    for graph in graphs:

        #baseline_results = []

        for seed in range(1, 11):
            
            with open(READ_DATA_PATHS["Results Data"] + f"/No Ablation/{model}/{graph}/{seed}/report.json") as f:
                baseline_result = json.load(f)["F1 - Positives"]
                #baseline_results.append(baseline_result)
                
                results["Graph"].append(graph)
                results["Method"].append(model)
                results["Seed"].append(seed)
                results["F1 - Positives"].append(baseline_result)



In [6]:
results = pd.DataFrame(results)
results

,Graph,Method,Seed,F1 - Positives
0,export,GCN,1,0.100380
1,export,GCN,2,0.104110
2,export,GCN,3,0.099051
3,export,GCN,4,0.107651
4,export,GCN,5,0.106133
...,...,...,...,...
195,multi-graph-export,MLP,6,0.898477
196,multi-graph-export,MLP,7,0.905371
197,multi-graph-export,MLP,8,0.901266
198,multi-graph-export,MLP,9,0.909091


In [7]:
for m in models:
    if m == "MLP":
        continue
    print(f"Model: {m}")
    for g in graphs:
        baseline = results[(results["Method"] == "MLP") & (results["Graph"] == g)]
        baseline_mean = baseline["F1 - Positives"].mean()
        subset = results[(results["Method"] == m) & (results["Graph"] == g)]
        mean = subset["F1 - Positives"].mean()
        _, p = mannwhitneyu(subset["F1 - Positives"], baseline["F1 - Positives"])
        print(f"  Graph: {g:<10} | Mean F1 - Positives: {mean:.4f} | Baseline: {baseline_mean:.4f} | Diff: {mean - baseline_mean:.4f}, p-value: {p:.4f}")

Model: GCN
  Graph: export     | Mean F1 - Positives: 0.1058 | Baseline: 0.0961 | Diff: 0.0097, p-value: 0.0211
  Graph: total      | Mean F1 - Positives: 0.1168 | Baseline: 0.0982 | Diff: 0.0186, p-value: 0.0013
  Graph: export-layered | Mean F1 - Positives: 0.1025 | Baseline: 0.1056 | Diff: -0.0032, p-value: 0.1405
  Graph: multi-graph-total | Mean F1 - Positives: 0.9220 | Baseline: 0.9079 | Diff: 0.0140, p-value: 0.0003
  Graph: multi-graph-export | Mean F1 - Positives: 0.9165 | Baseline: 0.9041 | Diff: 0.0124, p-value: 0.0002
Model: GAT
  Graph: export     | Mean F1 - Positives: 0.1239 | Baseline: 0.0961 | Diff: 0.0278, p-value: 0.0017
  Graph: total      | Mean F1 - Positives: 0.1307 | Baseline: 0.0982 | Diff: 0.0325, p-value: 0.0002
  Graph: export-layered | Mean F1 - Positives: 0.1153 | Baseline: 0.1056 | Diff: 0.0096, p-value: 0.0022
  Graph: multi-graph-total | Mean F1 - Positives: 0.9150 | Baseline: 0.9079 | Diff: 0.0071, p-value: 0.0211
  Graph: multi-graph-export | Mean F1 